# Gradio multi-modèles · DeepSeek 9B + Offrail 27B

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dbrckk/Colab/blob/main/DeepSeek_Offrail_Gradio_Colab.ipynb)

**Instructions :** dans Colab, sélectionner un GPU T4, exécuter toutes les cellules, ouvrir le lien Gradio et entrer les identifiants affichés par la dernière cellule. Choisir son modèle, cliquer sur **Charger / changer**, puis discuter. Après un changement de modèle, commencer une nouvelle conversation.

- [DeepSeek V4 Pro / Qwen3.5 9B GGUF Q4_K_M](https://huggingface.co/mradermacher/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-GGUF) : ~5,8 Go.
- [Offrail 27B](https://huggingface.co/sudoingx/Offrail-27B-Uncensored) : fichiers 12G (~7 Go) et 8G. Utilise le moteur ternaire spécial [sudoingX/llama.cpp, branche bonsai2](https://github.com/sudoingX/llama.cpp/tree/bonsai2).
- Un seul modèle chargé à la fois. Téléchargement et compilation à la demande. Les modèles ne tiennent pas forcément dans la RAM/VRAM offerte par Colab ; le GPU gratuit dépend des quotas.
- Le lien Gradio public est temporaire et protégé par mot de passe. Le moteur reste sur 127.0.0.1. L'historique Gradio se conserve dans le navigateur.


In [ ]:
# @title 1. Installer les dépendances
import sys, subprocess, os, shutil
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
 "gradio==6.5.1", "huggingface_hub>=0.35,<2", "requests>=2.32,<3"], check=True)
for command in ("git", "cmake", "g++", "curl"):
    if not shutil.which(command):
        raise RuntimeError("Commande requise absente : " + command)
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
print("Dépendances installées")


In [ ]:
# @title 2. Configurer les modèles et leur moteur
MODE = "GPU"  # @param ["GPU", "CPU"]
CONTEXTE = 4096  # @param [2048, 4096, 8192]
CACHE = "/content/llm-gradio"  # @param {type:"string"}

import atexit, hashlib, json, os, platform, re, shutil, socket, subprocess, tarfile
import threading, time
from pathlib import Path
import requests
from huggingface_hub import hf_hub_download

if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("Utilise Colab Linux x86-64")
GPU = MODE == "GPU"
if GPU:
    try:
        DEVICE = subprocess.check_output(["nvidia-smi", "--query-gpu=name",
            "--format=csv,noheader"], text=True, stderr=subprocess.DEVNULL).splitlines()[0].strip()
    except (OSError, subprocess.SubprocessError, IndexError) as exc:
        raise RuntimeError("Active le GPU T4 dans Colab ou sélectionne MODE=CPU") from exc
else:
    DEVICE = "CPU"
ROOT = Path(CACHE).expanduser().resolve()
ROOT.mkdir(parents=True, exist_ok=True)
SPECS = {
  "DeepSeek · 9B Q4_K_M": (
     "mradermacher/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-GGUF",
     "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.Q4_K_M.gguf", "official", "deepseek-9b"),
  "Offrail · 27B (12G qualité)": (
     "sudoingx/Offrail-27B-Uncensored", "Offrail-27B-12G.gguf", "bonsai", "offrail-27b"),
  "Offrail · 27B (8G mémoire)": (
     "sudoingx/Offrail-27B-Uncensored", "Offrail-27B-8G.gguf", "bonsai", "offrail-27b")
}
HASH_9B = "b4363b5710a29a03cab49bc10df92ca688f891eba1a63cae912e4d96f3fd7a65"
PROC = None
ACTIVE_FILE = None
PORT = None
LOCK = threading.RLock()
LOG = ROOT / "llama-server.log"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(8*1024*1024), b""):
            h.update(b)
    return h.hexdigest()

def official_binary():
    # Verified, stable upstream release already used by the original DeepSeek notebook.
    rel = "b12306"
    flavor = "cuda12.8" if GPU else "cpu"
    folder = ROOT / ("official-" + flavor)
    files = (
      [("llama-b12306-bin-ubuntu-cuda-12.8-x64.tar.gz",
        "9e9ee81e8a855e8765b2125946611df80d0a06bce8ec696f571fe81efe20224f"),
       ("cudart-llama-b12306-bin-ubuntu-cuda-12.8-x64.tar.gz",
        "2bb16fceea005fae7c6213a95fa1760bbcd799042a5ca88034ae439b6016bcfe")]
      if GPU else
      [("llama-b12306-bin-ubuntu-x64.tar.gz",
        "8ea4144d0698a960d943098ae87d027cfa219a44eed1302720b0f47bc30831e1")]
    )
    if not (folder / ".ready").exists():
        folder.mkdir(parents=True, exist_ok=True)
        for name, digest in files:
            archive = ROOT / name
            if not archive.exists() or sha256(archive) != digest:
                archive.unlink(missing_ok=True)
                subprocess.run(["curl", "-fL", "--retry", "3", "-o", str(archive),
                    f"https://github.com/ggml-org/llama.cpp/releases/download/{rel}/{name}"], check=True)
            if sha256(archive) != digest:
                raise RuntimeError("Archive llama.cpp corrompue : " + name)
            with tarfile.open(archive, "r:gz") as data:
                data.extractall(folder, filter="data")
        (folder / ".ready").touch()
    binary = next(folder.rglob("llama-server"), None)
    if binary is None:
        raise RuntimeError("llama-server absent de la distribution officielle")
    binary.chmod(binary.stat().st_mode | 0o111)
    dirs = sorted(set(str(p.parent) for p in folder.rglob("*.so*")))
    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = os.pathsep.join(dirs + [env.get("LD_LIBRARY_PATH", "")])
    return binary, env

def bonsai_binary():
    folder = ROOT / "bonsai2"
    binary = folder / "build" / "bin" / "llama-server"
    if not binary.is_file():
        if not (folder / ".git").exists():
            subprocess.run(["git", "clone", "--depth", "1", "-b", "bonsai2",
                "https://github.com/sudoingX/llama.cpp", str(folder)], check=True)
        cmake = ["cmake", "-S", str(folder), "-B", str(folder / "build"),
            "-DCMAKE_BUILD_TYPE=Release", "-DLLAMA_BUILD_SERVER=ON",
            "-DLLAMA_BUILD_EXAMPLES=OFF", "-DLLAMA_BUILD_TESTS=OFF",
            "-DGGML_CUDA=" + ("ON" if GPU else "OFF")]
        if GPU:
            nvcc = shutil.which("nvcc") or "/usr/local/cuda/bin/nvcc"
            if not Path(nvcc).exists():
                raise RuntimeError("nvcc CUDA absent : impossible de compiler Offrail sur GPU")
            cmake.append("-DCMAKE_CUDA_COMPILER=" + nvcc)
        subprocess.run(cmake, check=True)
        subprocess.run(["cmake", "--build", str(folder / "build"),
            "--target", "llama-server", "-j", str(min(4, os.cpu_count() or 2))], check=True)
    if not binary.is_file():
        raise RuntimeError("Échec de la compilation du moteur ternaire")
    env = os.environ.copy()
    env["GGML_CUDA_BATCH_INVARIANT"] = "1"
    return binary, env

def download_weights(spec):
    repo, filename, engine, alias = spec
    folder = ROOT / "models" / alias
    path = Path(hf_hub_download(repo_id=repo, filename=filename, local_dir=str(folder)))
    if path.stat().st_size < 1_000_000_000:
        raise RuntimeError("Téléchargement GGUF incomplet")
    if engine == "official":
        digest = HASH_9B
    else:
        sums = Path(hf_hub_download(repo_id=repo, filename="SHA256SUMS",
            local_dir=str(folder))).read_text()
        digest = next((s.split()[0] for s in sums.splitlines()
            if s.split() and s.split()[-1].lstrip("*") == filename), None)
        if not digest or not re.fullmatch("[a-fA-F0-9]{64}", digest):
            raise RuntimeError("Empreinte SHA256 manquante dans le dépôt Offrail")
    if sha256(path).lower() != digest.lower():
        path.unlink(missing_ok=True)
        raise RuntimeError("Checksum GGUF invalide : fichier supprimé")
    return path

def stop_model():
    global PROC, ACTIVE_FILE, PORT
    if PROC is not None and PROC.poll() is None:
        PROC.terminate()
        try:
            PROC.wait(timeout=10)
        except subprocess.TimeoutExpired:
            PROC.kill()
            PROC.wait()
    PROC, ACTIVE_FILE, PORT = None, None, None

def next_port():
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]

def start_model(spec, weights, binary, env):
    global PROC, ACTIVE_FILE, PORT
    stop_model()
    for gpu_layers in ([99, 35, 0] if GPU else [0]):
        port = next_port()
        args = [str(binary), "-m", str(weights), "--host", "127.0.0.1",
            "--port", str(port), "-c", str(CONTEXTE), "-ngl", str(gpu_layers),
            "--alias", spec[3], "--parallel", "1"]
        with LOG.open("a") as log:
            log.write("\\nStarting " + spec[1] + " layers=" + str(gpu_layers) + "\\n")
            log.flush()
            proc = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT, env=env)
        started = time.monotonic()
        while time.monotonic() - started < 150:
            if proc.poll() is not None:
                break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    PROC, ACTIVE_FILE, PORT = proc, spec[1], port
                    return
            except requests.RequestException:
                pass
            time.sleep(2)
        if proc.poll() is None:
            proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
    raise RuntimeError(f"Chargement impossible ; voir le journal {LOG}")

def load_model(choice, progress=None):
    with LOCK:
        spec = SPECS[choice]
        if PROC is not None and PROC.poll() is None and ACTIVE_FILE == spec[1]:
            return "Modèle prêt : " + choice
        if progress:
            progress(0.06, desc="Préparation du moteur")
        binary, env = official_binary() if spec[2] == "official" else bonsai_binary()
        if progress:
            progress(0.16, desc="Téléchargement et contrôle du GGUF")
        weights = download_weights(spec)
        if progress:
            progress(0.80, desc="Chargement du modèle en RAM/VRAM")
        start_model(spec, weights, binary, env)
        return "Modèle prêt : " + choice + " | " + DEVICE

def api():
    if PROC is None or PROC.poll() is not None or PORT is None:
        raise RuntimeError("Le moteur n'est pas actif")
    return f"http://127.0.0.1:{PORT}"

atexit.register(stop_model)
print("Choix disponibles :", list(SPECS))
print("Matériel :", DEVICE, "| cache :", ROOT)


In [ ]:
# @title 3. Lancer Gradio et basculer entre les deux modèles
import gradio as gr
import secrets, json
ON_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
PASSWORD = secrets.token_urlsafe(15)
SYS_PROMPT = "Tu es un assistant clair et précis. Réponds dans la langue de l'utilisateur."

def select_model(choice, progress=gr.Progress()):
    try:
        return load_model(choice, progress)
    except Exception as exc:
        return f"Échec du chargement : {type(exc).__name__} : {str(exc)[:400]} (journal : {LOG})"

def answer(message, history, selected, prompt, temperature, top_p, max_output):
    if not message or not message.strip():
        return
    yield "Préparation du modèle…"
    try:
        load_model(selected)
        endpoint = api()
    except Exception as exc:
        raise gr.Error(f"Modèle indisponible : {exc}") from exc
    older = []
    for item in (history or [])[-18:]:
        if isinstance(item, dict) and item.get("role") in ("user", "assistant") and isinstance(item.get("content"), str):
            older.append({"role": item["role"], "content": item["content"]})
    # Supprime les échanges anciens lorsque le contexte devient trop volumineux.
    while older and sum(len(row["content"]) for row in older) > CONTEXTE * 2 - 1500:
        older.pop(0)
    messages = ([{"role": "system", "content": prompt}] if prompt.strip() else [])
    messages += older + [{"role": "user", "content": message.strip()}]
    response = None
    result = ""
    try:
        response = requests.post(endpoint + "/v1/chat/completions",
            json={"model": SPECS[selected][3], "messages": messages,
                  "temperature": float(temperature), "top_p": float(top_p),
                  "max_tokens": int(max_output), "stream": True},
            stream=True, timeout=(15,180), headers={"Accept": "text/event-stream"})
        response.raise_for_status()
        for raw in response.iter_lines():
            if not raw:
                continue
            item = raw.decode("utf-8", "replace")
            if not item.startswith("data: "):
                continue
            data = item[6:]
            if data == "[DONE]":
                break
            try:
                event = json.loads(data)
            except json.JSONDecodeError:
                continue
            if event.get("error"):
                raise gr.Error(str(event["error"])[:300])
            for candidate in event.get("choices") or []:
                delta = (candidate.get("delta") or {}).get("content") or ""
                if isinstance(delta, str) and delta:
                    result += delta
                    yield result
    except requests.RequestException as exc:
        raise gr.Error(f"Connexion interrompue : {exc}. Journal : {LOG}") from exc
    finally:
        if response is not None:
            response.close()
    if not result.strip():
        yield "Aucune réponse textuelle. Essaie avec d'autres réglages."

CSS = """
.gradio-container {max-width: 1150px !important;}
#hero {padding: 22px; border-radius: 18px; border: 1px solid #2b405d;
 background: linear-gradient(120deg,#142946,#101827); margin-bottom: 14px;}
#hero h1 {margin:0; color:#eef6ff; font-size:clamp(24px,4vw,36px);}
#hero p {margin:8px 0 0; color:#bcd0e6;}
@media(max-width:650px){.gradio-container{padding:7px!important;}#hero{padding:14px;}}
"""
if "webapp" in globals():
    try:
        webapp.close()
    except Exception:
        pass
with gr.Blocks(title="DeepSeek + Offrail · Colab", analytics_enabled=False) as webapp:
    gr.HTML('<div id="hero"><h1>DeepSeek & Offrail</h1><p>Chat GGUF sur Colab · Gradio</p></div>')
    with gr.Row():
        choose = gr.Dropdown(choices=list(SPECS), value=list(SPECS)[0],
                             label="Modèle IA", scale=5)
        button = gr.Button("Charger / changer", variant="primary", scale=2)
    info = gr.Markdown(f"Matériel : **{DEVICE}**. Sélectionne le modèle et clique sur Charger.")
    with gr.Accordion("Réglages avancés", open=False):
        instructions = gr.Textbox(label="Instructions système", value=SYS_PROMPT, lines=2)
        temperature = gr.Slider(0, 1.5, value=0.7, step=0.05, label="Température")
        top_p = gr.Slider(0.1, 1, value=0.9, step=0.05, label="Top-p")
        max_output = gr.Slider(128, min(2048, CONTEXTE - 256),
                 value=min(1024, CONTEXTE - 256), step=128, label="Max tokens")
    conversation = gr.Chatbot(layout="bubble",
                 height="57vh", sanitize_html=True, show_label=False,
                 placeholder="Sélectionne un modèle puis envoie un message.")
    gr.ChatInterface(fn=answer, chatbot=conversation,
        textbox=gr.Textbox(placeholder="Écris ton message…", lines=2, max_lines=6,
                           show_label=False, submit_btn="Envoyer", stop_btn="Arrêter"),
        additional_inputs=[choose, instructions, temperature, top_p, max_output],
        additional_inputs_accordion=gr.Accordion(visible=False),
        save_history=True, concurrency_limit=1, show_progress="minimal")
    button.click(select_model, inputs=[choose], outputs=[info],
                 concurrency_id="llm", concurrency_limit=1)
    choose.change(lambda name: f"**{name}** sélectionné. Clique sur Charger / changer.",
                  inputs=[choose], outputs=[info], show_progress="hidden")
    gr.Markdown("Historique enregistré dans le navigateur. Après un changement de modèle, ouvre une nouvelle conversation.")
print("Utilisateur Gradio : chat")
if ON_COLAB:
    print("Mot de passe Gradio :", PASSWORD)
_, local_url, public_url = webapp.queue(default_concurrency_limit=1).launch(
    share=ON_COLAB, inline=not ON_COLAB,
    auth=("chat", PASSWORD) if ON_COLAB else None,
    show_error=True, prevent_thread_lock=True, enable_monitoring=False,
    theme=gr.themes.Soft(primary_hue="cyan", neutral_hue="slate"), css=CSS)
print("Lien Gradio :", public_url or local_url)
